# Data Analysis CABINET-XX, 50 SEW Motor (6 Days Data)

## Functions

In [ ]:
import pandas as pd
import numpy as np
from sklearn.preprocessing import StandardScaler
from sklearn.decomposition import PCA
from sklearn.linear_model import LinearRegression

def clean_columns(sub_df):
    """
    Remove constant columns and raw duplicates (keep only engineering units).
    Returns list of columns ready for analysis.
    """
    numeric_cols = sub_df.select_dtypes(include=[np.number]).columns.tolist()
    
    # Remove constants
    constant_cols = [col for col in numeric_cols if sub_df[col].nunique() <= 1]
    numeric_cols = [c for c in numeric_cols if c not in constant_cols]
    
    # Remove raw columns that have engineering-unit versions
    raw_to_remove = []
    for col in numeric_cols:
        if not any(u in col for u in ['_rpm', '_%', '_degC', '_V', '_A', '_Hz', '_scaled']):
            scaled_versions = [c for c in numeric_cols if c.startswith(col + '_') and c != col]
            if scaled_versions:
                raw_to_remove.append(col)
    
    analysis_cols = [c for c in numeric_cols if c not in raw_to_remove]
    
    return analysis_cols, constant_cols, raw_to_remove


def correlation_analysis(sub_df, analysis_cols, threshold=0.9):
    """
    Compute correlation matrix and identify clusters of redundant features.
    threshold: correlation above which two features are considered redundant.
    
    Returns:
        corr_matrix: Correlation matrix DataFrame
        clusters: list of lists (groups of correlated features)
        recommended_keep: list of features to keep (1 per cluster)
    """
    X = sub_df[analysis_cols]
    corr_matrix = X.corr()
    
    # Correlation-based clustering
    visited = set()
    clusters = []
    
    for i, col_i in enumerate(analysis_cols):
        if col_i in visited:
            continue
        cluster = [col_i]
        visited.add(col_i)
        for j, col_j in enumerate(analysis_cols):
            if col_j not in visited and abs(corr_matrix.loc[col_i, col_j]) > threshold:
                cluster.append(col_j)
                visited.add(col_j)
        clusters.append(cluster)
    
    # For each cluster, keep the feature with highest variance (most informative)
    recommended_keep = []
    for cluster in clusters:
        if len(cluster) == 1:
            recommended_keep.append(cluster[0])
        else:
            variances = {feat: sub_df[feat].std() for feat in cluster}
            best = max(variances, key=variances.get)
            # Alternative approach:
            #avg_corr = {feat: corr_matrix.loc[feat, cluster].abs().mean() for feat in cluster}
            #best = max(avg_corr, key=avg_corr.get)
            recommended_keep.append(best)
    
    return corr_matrix, clusters, recommended_keep

## Execution

In [ ]:
import pandas as pd
import numpy as np
from sklearn.preprocessing import StandardScaler
from sklearn.decomposition import PCA
import matplotlib.pyplot as plt

# === LOAD DATA ===
df = pd.read_csv("../data/sample_data.csv")
print(f"Dataset shape: {df.shape}")
print(f"Unique drives: {df['drive_ip'].nunique()}")
print(f"Rows per drives: ~{len(df) // df['drive_ip'].nunique()}")
print(f"Timestamp range: {df['timestamp'].min()} → {df['timestamp'].max()}")

# === SPLIT BY DRIVE IP ===
drive_groups = {ip: group.reset_index(drop=True) for ip, group in df.groupby('drive_ip')}

print(f"Created {len(drive_groups)} mini-datasets:")
for ip, sub_df in drive_groups.items():
    print(f"  {ip}: {sub_df.shape[0]} rows x {sub_df.shape[1]} columns")

In [ ]:
# === CORRELATION ANALYSIS FOR EACH DRIVE ===
# (make sure you ran cells 1 and 2 first)
drive_groups = {ip: group.reset_index(drop=True) for ip, group in df.groupby('drive_ip')}

results = {}

for ip, sub_df in drive_groups.items():
    analysis_cols, constant_cols, raw_removed = clean_columns(sub_df)
    corr_matrix, clusters, recommended_keep = correlation_analysis(sub_df, analysis_cols)
    
    results[ip] = {
        'analysis_cols': analysis_cols,
        'constant_cols': constant_cols,
        'corr_matrix': corr_matrix,
        'clusters': clusters,
        'recommended_keep': recommended_keep,
        'n_features_original': len(analysis_cols),
        'n_features_reduced': len(recommended_keep),
    }

# Summary
print(f"{'Drive IP':<18} {'Features':<12} {'Clusters':<12} {'Keep':<8}")
print("-" * 50)
for ip in sorted(results.keys()):
    r = results[ip]
    print(f"{ip:<18} {r['n_features_original']:<12} {len(r['clusters']):<12} {r['n_features_reduced']:<8}")

In [ ]:
# === RECOMMENDED FEATURES FOR EACH DRIVE ===
print("=" * 70)
print("FEATURES TO KEEP PER DRIVE (1 per cluster, |r| > 0.9)")
print("=" * 70)

for ip in sorted(results.keys()):
    r = results[ip]
    keep = r['recommended_keep']
    print(f"\n  {ip} ({r['n_features_original']} → {r['n_features_reduced']} features):")
    for j, feat in enumerate(keep, 1):
        print(f"    {j:>2}. {feat}")

# === COMPARISON: ARE THEY THE SAME ACROSS ALL DRIVES? ===
from collections import Counter

print(f"\n\n{'='*70}")
print("CROSS-DRIVE COMPARISON — RECOMMENDED FEATURE FREQUENCY")
print(f"{'='*70}\n")

keep_freq = Counter(feat for r in results.values() for feat in r['recommended_keep'])
all_keep_sets = [set(r['recommended_keep']) for r in results.values()]
common_keep = set.intersection(*all_keep_sets)

print(f"Features COMMON to all {len(results)} drives: {len(common_keep)}\n")
print(f"{'Feature':<35} {'Drive count':<15} {'%':<8} {'Common?'}")
print("-" * 65)
for feat, count in keep_freq.most_common():
    pct = count / len(results) * 100
    marker = "✓ ALL" if feat in common_keep else ""
    print(f"{feat:<35} {count}/{len(results):<12} {pct:>5.0f}%   {marker}")

## Correlation Matrices

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.colors as mcolors
import seaborn as sns
cmap = sns.color_palette("rocket_r", as_cmap=True)

# === FAST VERSION — skip seaborn, use imshow directly ===

sorted_ips = sorted(results.keys())
n_drives = len(sorted_ips)
ncols = 8
nrows = int(np.ceil(n_drives / ncols))

fig, axes = plt.subplots(
    nrows, ncols,
    figsize=(4 * ncols, 3.5 * nrows),
    constrained_layout=True
)
axes_flat = axes.flatten()

cmap = plt.cm.YlOrRd      # yellow → red (classic for correlations)  # or plt.cm.YlOrRd
norm = mcolors.Normalize(vmin=0, vmax=1)

for idx, ip in enumerate(sorted_ips):
    ax = axes_flat[idx]
    corr = results[ip]['corr_matrix'].abs().values  # numpy array = fast

    # imshow is MUCH faster than sns.heatmap (no Text objects)
    im = ax.imshow(corr, cmap=cmap, norm=norm, aspect='equal', interpolation='nearest')

    drive_id = ip.split('.')[-1]
    ax.set_title(f"Drive .{drive_id}", fontsize=18, fontweight='bold', pad=6)
    ax.set_xticks([])
    ax.set_yticks([])

# Hide unused subplots
for idx in range(n_drives, len(axes_flat)):
    axes_flat[idx].set_visible(False)

# Shared colorbar
cbar = fig.colorbar(
    plt.cm.ScalarMappable(cmap=cmap, norm=norm),
    ax=axes_flat[:n_drives].tolist(),
    location='right',
    fraction=0.025,
    pad=0.02
    #label='|Correlation|'
    #fontsize=16
)
cbar.ax.tick_params(labelsize=16)

fig.suptitle(
    "Absolute Correlation Matrix per Drive\n"
    "(0 = independent, 1 = redundant)",
    fontsize=18,
    fontweight='bold'
)

#plt.savefig("correlation_abs_all_drives_grid_last.png", dpi=150, bbox_inches='tight')
plt.show()

# $R^2$ Analysis

In [ ]:
# === PLOT: MEAN R² vs NUMBER OF RETAINED FEATURES ===
from sklearn.linear_model import LinearRegression

# Ordina features per frequenza (le più comuni prima)
sorted_features = [feat for feat, count in keep_freq.most_common()]
max_features = len(sorted_features)

# Compute R² for each possible n_features_retained
n_range = range(1, max_features + 1)
r2_mean_list = []
r2_min_list = []
r2_max_list = []

for n in n_range:
    top_n = sorted_features[:n]
    r2_per_drive = []
    
    for ip in sorted(results.keys()):
        sub_df = drive_groups[ip]
        analysis_cols = results[ip]['analysis_cols']
        
        kept = [f for f in top_n if f in analysis_cols]
        removed = [f for f in analysis_cols if f not in kept]
        
        if not removed or not kept:
            r2_per_drive.append(100.0)
            continue
        
        X_kept = sub_df[kept].values
        r2_list = []
        for col in removed:
            y = sub_df[col].values
            reg = LinearRegression().fit(X_kept, y)
            r2 = max(reg.score(X_kept, y), 0)
            r2_list.append(r2)
        
        r2_per_drive.append(np.mean(r2_list) * 100)
    
    r2_mean_list.append(np.mean(r2_per_drive))
    r2_min_list.append(np.min(r2_per_drive))
    r2_max_list.append(np.max(r2_per_drive))

# Plot
fig, ax = plt.subplots(figsize=(12, 6))

ax.plot(list(n_range), r2_mean_list, 'b-o', markersize=5, label='Mean R² (avg across drives)')
ax.plot(list(n_range), r2_min_list, 'r--^', markersize=4, label='Mean R² (worst drive)')
ax.plot(list(n_range), r2_max_list, 'g--v', markersize=4, label='Mean R² (best drive)')

ax.fill_between(list(n_range), r2_min_list, r2_max_list, alpha=0.1, color='blue')
ax.axhline(y=95, color='gray', linestyle=':', label='95% threshold')
ax.axhline(y=90, color='gray', linestyle='--', alpha=0.5, label='90% threshold')

ax.set_xlabel("Number of Retained Features", fontsize=12)
ax.set_ylabel("Mean R² (%)", fontsize=12)
ax.set_title("R² vs Number of Retained Features (across all drives)", fontsize=14)
ax.set_xticks(list(n_range))
ax.set_ylim(0, 102)
ax.legend(loc='lower right')
ax.grid(True, alpha=0.3)

plt.tight_layout()
#plt.savefig("r2_vs_n_features_best.png", dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

# === R² vs NUMBER OF RETAINED FEATURES (ALL DRIVES) ===

plt.rcParams.update({
    'font.family': 'sans-serif',
    'font.size': 18,
    'axes.titlesize': 18,
    'axes.titleweight': 'bold',
    'axes.labelsize': 16,
})

fig, ax = plt.subplots(figsize=(13, 6.5))

# Main lines
ax.plot(list(n_range), r2_mean_list, 'o-',
        color='#1f77b4', markersize=6, linewidth=2.2,
        label='Mean R² (avg across drives)', zorder=5)
ax.plot(list(n_range), r2_min_list, '^--',
        color='#d62728', markersize=5, linewidth=1.4,
        label='Mean R² (worst drive)', zorder=4)
ax.plot(list(n_range), r2_max_list, 'v--',
        color='#2ca02c', markersize=5, linewidth=1.4,
        label='Mean R² (best drive)', zorder=4)

# Shaded envelope between min and max
ax.fill_between(list(n_range), r2_min_list, r2_max_list,
                alpha=0.08, color='#1f77b4', label='_nolegend_')

# Threshold lines
ax.axhline(y=95, color='#555555', linestyle=':', linewidth=1.2, label='95% threshold')
ax.axhline(y=90, color='#999999', linestyle='--', linewidth=1.0, alpha=0.7, label='90% threshold')

# Annotate thresholds on the right margin
ax.text(list(n_range)[-1] + 0.3, 95.4, '95%', fontsize=15, color='#555555', va='bottom')
ax.text(list(n_range)[-1] + 0.3, 90.4, '90%', fontsize=15, color='#999999', va='bottom')

# Axes
ax.set_xlabel("Number of Retained Features", fontsize=18, labelpad=8)
ax.set_ylabel("Mean R² (%)", fontsize=18, labelpad=8)
ax.set_title("Reconstruction Accuracy (R²) vs. Number of Retained Features\n"
             "Evaluated across all drives", fontsize=18, pad=12)
ax.set_xticks(list(n_range))
ax.set_ylim(0, 102)
ax.set_xlim(min(n_range) - 0.5, max(n_range) + 0.5)

# Legend
ax.legend(loc='lower right', fontsize=18, framealpha=0.9, edgecolor='lightgray')

# Grid
ax.grid(True, alpha=0.25, linestyle='-', linewidth=0.5)
ax.set_axisbelow(True)

# Remove top/right spines for cleaner look
for spine in ['top', 'right']:
    ax.spines[spine].set_visible(False)

plt.tight_layout()
#plt.savefig("r2_vs_n_features_best_1.png", dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
print(sorted_features)

In [ ]:
for i in range(len(sorted_features)):
    print(i+1,":", sorted_features[i])

In [ ]:
# === EXPLAINED VARIANCE: N FEATURES vs ALL ===
from sklearn.preprocessing import StandardScaler
from sklearn.decomposition import PCA

n_features_retained = 13  # <-- CHANGE THIS NUMBER

# Take the top N most frequent features across all drives
top_n_features = [feat for feat, count in keep_freq.most_common(n_features_retained)]

print(f"Top {n_features_retained} selected features:")
for i, feat in enumerate(top_n_features, 1):
    print(f"  {i:>2}. {feat}")

# For each drives: how much of the REMOVED features is reconstructable from the RETAINED ones?
print(f"\n{'='*70}")
print(f"EXPLAINED VARIANCE — Mean R² of removed features predicted from retained ones")
print(f"{'='*70}\n")
print(f"{'Drive IP':<18} {'Mean R²':<12} {'Min R²':<12} {'Max R²':<12}")
print("-" * 55)

from sklearn.linear_model import LinearRegression

r2_results = {}
for ip in sorted(results.keys()):
    sub_df = drive_groups[ip]
    analysis_cols = results[ip]['analysis_cols']
    
    # Retained and removed features
    kept = [f for f in top_n_features if f in analysis_cols]
    removed = [f for f in analysis_cols if f not in kept]
    
    if not removed:
        r2_results[ip] = 100.0
        continue
    
    X_kept = sub_df[kept].values
    r2_list = []
    
    for col in removed:
        y = sub_df[col].values
        reg = LinearRegression().fit(X_kept, y)
        r2 = reg.score(X_kept, y)
        r2_list.append(max(r2, 0))  # clip a 0 se negativo
    
    r2_mean = np.mean(r2_list) * 100
    r2_min = np.min(r2_list) * 100
    r2_max = np.max(r2_list) * 100
    r2_results[ip] = r2_mean
    print(f"{ip:<18} {r2_mean:<12.1f}% {r2_min:<12.1f}% {r2_max:<12.1f}%")

print(f"\n{'='*70}")
print(f"SUMMARY")
print(f"{'='*70}")
print(f"  Features retained: {n_features_retained}/{len(analysis_cols)}")
print(f"  Global mean R²: {np.mean(list(r2_results.values())):.1f}%")
print(f"  → With {n_features_retained} features you can reconstruct {np.mean(list(r2_results.values())):.1f}%")
print(f"    of the removed features (on average)")

In [ ]:
# === MEMORY ESTIMATION (dynamic from analysis results) ===

# Use the top-N features from the frequency analysis
n_selected = 13  # <-- adjust as needed
selected_features = [feat for feat, count in keep_freq.most_common(n_selected)]

def fmt_bytes(b):
    for unit in ['B', 'KB', 'MB', 'GB']:
        if b < 1024:
            return f"{b:.1f} {unit}"
        b /= 1024
    return f"{b:.1f} TB"

# Estimate for the full fleet
n_motors = 500
sample_interval_sec = 5
hours = 24
records_per_motor = (hours * 3600) // sample_interval_sec
total_records = n_motors * records_per_motor

bytes_per_feature = 8  # float64
bytes_per_record_full = len(analysis_cols) * bytes_per_feature
bytes_per_record_reduced = n_selected * bytes_per_feature

print(f"{'='*60}")
print(f"MEMORY ESTIMATION — {n_motors} motors, {hours}h, {sample_interval_sec}s interval")
print(f"{'='*60}")
print(f"\n  Records per motor/day: {records_per_motor:,}")
print(f"  Total records/day:     {total_records:,}")
print(f"\n  Full feature set ({len(analysis_cols)} features):")
print(f"    Per record:  {fmt_bytes(bytes_per_record_full)}")
print(f"    Per day:     {fmt_bytes(total_records * bytes_per_record_full)}")
print(f"\n  Reduced feature set ({n_selected} features):")
print(f"    Per record:  {fmt_bytes(bytes_per_record_reduced)}")
print(f"    Per day:     {fmt_bytes(total_records * bytes_per_record_reduced)}")
print(f"\n  Savings: {(1 - n_selected/len(analysis_cols))*100:.0f}% reduction")
print(f"\n  Selected features: {selected_features}")
